# Text to Image & Video Google Colab GPU Backend

Notebook này cho phép khởi chạy Backend chuyển đổi **Text Prompt -> Image -> Video (MP4)** trên GPU miễn phí của Google Colab.

### Hướng dẫn sử dụng:
1. Truy cập menu **Runtime** -> **Change runtime type** -> Chọn GPU làm Hardware accelerator (T4 / A100).
2. Nhấn nút **Run all (Chạy tất cả)** hoặc chạy từng ô code bên dưới.
3. Chờ cho đến khi Cloudflare Tunnel xuất hiện liên kết công khai dạng `https://xxx.trycloudflare.com`.
4. Sử dụng liên kết đó để gửi API tạo video từ văn bản.

### Các API Endpoints:
- **Tạo Video từ Text** : `POST {URL}/generate` với JSON body `{"prompt": "A beautiful sunset over mountains"}`
- **Kiểm tra trạng thái** : `GET {URL}/status/{task_id}`
- **Tải Video MP4**     : `GET {URL}/download/{task_id}`
- **Xem Ảnh PNG**       : `GET {URL}/image/{task_id}`

In [ ]:
#@title 1. Cài đặt môi trường và các thư viện cần thiết
import os
import sys

print("--- 1. Đang chuẩn bị mã nguồn ứng dụng... ---")
if not os.path.exists("/content/imagetovideo"):
    !git clone https://github.com/akavipno01/imagetovideo.git /content/imagetovideo

%cd /content/imagetovideo

print("\n--- 2. Đang cài đặt thư viện PyTorch, Diffusers, OpenCV & FastAPI... ---")
!pip install -q -r backend/requirements.txt

print("\n--- 3. Tải Cloudflare Tunnel (cloudflared)... ---")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("\nHoàn tất chuẩn bị môi trường GPU!")

In [ ]:
#@title 2. Khởi chạy API Backend & Đường truyền Cloudflare Tunnel
import subprocess
import time
import re

data_dir = "/content/data"
os.makedirs(os.path.join(data_dir, "outputs", "images"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "outputs", "videos"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "temp"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "models"), exist_ok=True)

print("Đang khởi động Text-to-Video FastAPI Backend...")
backend_process = subprocess.Popen(
    ["python", "run.py"],
    cwd="/content/imagetovideo/backend",
    env={
        **os.environ,
        "TEXT_TO_VIDEO_PORT": "3930",
        "TEXT_TO_VIDEO_DATA_DIR": data_dir
    }
)

time.sleep(3)

print("Đang khởi tạo đường truyền kết nối công khai qua Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:3930"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

colab_url = None
try:
    while True:
        line = tunnel_process.stdout.readline()
        if not line:
            break
        if "trycloudflare.com" in line or "error" in line.lower() or "tunnel" in line.lower():
            print("[Cloudflared]", line.strip())
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            colab_url = match.group(0)
            print("\n" + "="*65)
            print(f"🎉 KHỞI CHẠY GOOGLE COLAB TEXT-TO-VIDEO BACKEND THÀNH CÔNG!")
            print(f"🔗 Địa chỉ API công khai của bạn là:")
            print(f"   {colab_url}")
            print("\nHướng dẫn gọi API:")
            print(f"1. Tạo Video từ Text: POST {colab_url}/generate")
            print(f"   Header: Content-Type: application/json")
            print(f"   Body  : {{\"prompt\": \"A majestic dragon flying over snowy mountains, 4k\"}}")
            print(f"2. Kiểm tra tiến độ : GET {colab_url}/status/{{task_id}}")
            print(f"3. Tải Video MP4    : GET {colab_url}/download/{{task_id}}")
            print(f"4. Xem Ảnh PNG      : GET {colab_url}/image/{{task_id}}")
            print("="*65 + "\n")
            
    backend_process.wait()
except KeyboardInterrupt:
    print("\nĐang dừng hệ thống...")
    tunnel_process.terminate()
    backend_process.terminate()
    print("Đã dừng.")
